In [0]:
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

def ingest_to_bronze(
    spark: SparkSession, 
    schema: StructType, 
    source_path: str, 
    target_path: str
) -> DataFrame:
    """
    Reads a raw CSV file, adds ingestion metadata, and saves it as a Delta table.
    """
    # 1. Read the raw data
    df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .schema(schema)
        .csv(source_path)
    )
    
    # 2. Add ingestion timestamp
    df = df.withColumn("ingest_timestamp", current_timestamp())
    
    # 3. Write to Delta in overwrite mode
    df.write.mode("overwrite").format("delta").save(target_path)
    
    # Useful for debugging
    print(f"Successfully ingested data to {target_path}")

    return df